#### 프롬프트 엔지니어링
1. 명확하고 구체적으로 작성
    - 적당한 길이로 답변 X -> 100자 내외로 답변
    - 예쁘게 답변해 -> 예시를 줘가면서 답변하라고 유도 -> 퓨샷
2. 명확하게 작성하는 것이 어렵다면(few shot)
    - 너는 경상도 사투리로 답변하는 친구야 - 물음에 경상도 사투리로 답변
    - ex) - 표준어 : 오늘 밥먹었니
          - 경상도 : 오늘 밥먹었나
          - 표준어 : 무엇을 떨어뜨렸다
          - 경상도 : 무엇을 널짯다
          - 표준어 : 많이 힘들다
          - 경상도 : 대다
3. 출력 형식 : 결과물을 json 받는게 좋습니다
    - 나중에 결과물을 저장하기 좋게 -> 데이터베이스에 넣거나 처리하기 좋게 key : value 이런 형식으로 받는게 좋다
    

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv() 
client = OpenAI()

In [ ]:
# response = client.chat.completions.create(

#     model= "gpt-5.6-luna",
#     messages=[{

#         "role" : "user",
#         "content" : " 파이썬의 장단점 추천해줘 3줄 내외로"
#     }]
# )

# print(response)

In [9]:
# 요청 함수를 만들어 재사용 -시스템 프롬프트를 적용한 단일 턴 대화(멀티 턴 X)
def ask(user,system=None,model="gpt-5.6-luna"):
    messages = []
    if system:
            messages.append({"role" : "system", "content":system}) # 시스템프롬프트 넣기
    messages.append({"role":"user", "content": user}) # 실제로 질문하는 유저 메시지 넣기
    response = client.chat.completions.create(  #응답요청
          model=model,messages=messages
    )
    return response.choices[0].message.content

In [10]:
question = "간단한 이력서를 작성해줘"
result = ask(question)
print(result)

아래 형식으로 간단한 이력서를 작성할 수 있습니다.

# 이력서

## 1. 인적사항
- 이름: 홍길동
- 연락처: 010-0000-0000
- 이메일: example@email.com
- 주소: 서울특별시 ○○구

## 2. 학력
- 20XX.03 ~ 20XX.02 ○○대학교 ○○학과 졸업
- 20XX.03 ~ 20XX.02 ○○고등학교 졸업

## 3. 경력사항
- 20XX.03 ~ 20XX.12 ○○회사 / ○○팀 / 사원
  - 담당 업무: 고객 응대, 문서 작성, 자료 관리
- 20XX.01 ~ 20XX.02 ○○인턴
  - 담당 업무: 업무 보조 및 데이터 정리

## 4. 자격 및 교육
- 컴퓨터활용능력 2급
- 워드프로세서
- 운전면허 2종 보통

## 5. 보유 역량
- 문서 작성 및 자료 정리
- 원활한 의사소통 능력
- MS Office 활용 가능
- 책임감 있고 성실한 업무 태도

## 6. 자기소개
성실하고 책임감 있는 자세로 맡은 업무를 끝까지 수행합니다. 원활한 의사소통과 빠른 업무 습득 능력을 바탕으로 조직에 기여하겠습니다.

원하시면 **지원 직무, 학력, 경력, 자격증, 이름**을 알려주시면 실제 제출용 이력서로 다듬어드릴게요.


In [13]:
import os

In [ ]:
# 파일 저장해서 확인해보기
with open("result.md", "w", encoding="utf-8-sig") as file:
    file.write(result) 

물론입니다. 아래 양식에 정보를 채워주시면 깔끔한 이력서로 정리해드릴게요.

---

# 이력서

## 1. 인적사항
- 이름:
- 연락처:
- 이메일:
- 거주지:

## 2. 지원 분야
- 지원 직무:
- 희망 근무 형태:

## 3. 경력사항
### 회사명 / 직무
- 근무 기간:
- 담당 업무:
- 주요 성과:

## 4. 학력사항
- 학교명:
- 전공:
- 재학 기간:
- 졸업 여부:

## 5. 자격증 및 교육
- 자격증명 / 취득일:
- 교육명 / 수료일:

## 6. 보유 역량
- 컴퓨터 활용:
- 외국어:
- 기타 역량:

## 7. 자기소개
안녕하세요. 저는 __________ 분야에 관심과 경험을 가진 인재입니다.  
__________ 경험을 통해 __________ 역량을 쌓았으며, 이를 바탕으로 회사에 기여하고 싶습니다.

---

이름, 지원 직무, 학력, 경력, 자격증 등의 정보를 보내주시면 완성된 이력서 형태로 작성해드리겠습니다.


In [14]:
question = "간단한 이력서 작성해줘"

system_prompt = """
너는 10년 이상의 경력을 가진 전문 취업 컨설턴트이자
이력서·자기소개서 작성 전문가야.

사용자의 경험, 역량, 경력, 학력, 프로젝트 정보를 바탕으로
지원 직무에 적합한 이력서를 작성해줘.

사용자가 정보를 충분히 제공하지 않은 경우에는 내용을 임의로
만들어내지 말고, 사실이 확인되지 않은 부분은 '[추가 입력 필
요]'로
표시해줘.

이력서는 간결하고 전문적인 한국어로 작성해.
추상적인 표현이나 과도한 수식어는 줄이고, 지원 직무와 관련성이
높은 경험과 역량을 우선적으로 강조해.

가능한 경우 경험을 다음 구조로 작성해:
- 상황 또는 문제
- 수행한 행동
- 결과 또는 성과

성과는 가능한 경우 기간, 비율, 인원, 금액, 횟수 등 구체적인
수치를 사용해. 단, 사용자가 제공하지 않은 수치를 절대 지어내지
마.

사용자의 경력이 부족한 경우에도 교육, 프로젝트, 아르바이트,
동아리, 봉사활동, 개인 학습 경험을 직무 역량과 연결해서 작성해.

사용자가 '간단한 이력서'라고 요청했으므로 다음 항목을 중심으로
간결하게 작성해줘.

# 이력서

## 1. 기본 정보
- 이름: [추가 입력 필요]
- 연락처: [추가 입력 필요]
- 이메일: [추가 입력 필요]
- 거주 지역: [추가 입력 필요]
- 포트폴리오 또는 GitHub: [추가 입력 필요]

## 2. 지원 분야
- 희망 직무: [추가 입력 필요]
- 희망 산업 또는 회사: [추가 입력 필요]

## 3. 핵심 요약
지원자의 경력과 강점을 3~5문장으로 요약해.
지원 직무와 연결되는 역량이 드러나도록 작성해.

## 4. 핵심 역량
지원 직무와 관련된 역량을 4~6개 작성해.
각 역량은 실제 경험과 연결해서 설명해.

## 5. 경력 및 프로젝트
각 경험을 다음 형식으로 작성해.

### 경험 또는 프로젝트명
- 기간: [추가 입력 필요]
- 역할: [추가 입력 필요]
- 주요 업무:
- 사용 기술 또는 도구:
- 결과 및 성과:

## 6. 학력
- 학교명: [추가 입력 필요]
- 전공: [추가 입력 필요]
- 기간: [추가 입력 필요]
- 졸업 여부: [추가 입력 필요]

## 7. 자격증 및 교육
- 자격증 또는 교육명:
- 취득일 또는 수료일:
- 기관:

## 8. 보완 사항
이력서를 더 완성도 있게 만들기 위해 추가로 필요한 정보를
간단히 안내해줘.

이력서 내용은 Markdown 형식으로 작성해줘.
"""


# 시스템 프롬프트와 질문을 함께 전달
result = ask(
    question,
    system=system_prompt
)


# 결과를 Markdown 파일로 저장
file_path = os.path.abspath("result.md")

with open(file_path, "w", encoding="utf-8-sig") as file:
    file.write(result)


# 결과 확인
print(result)
print(f"\n파일 저장 완료: {file_path}")

# 이력서

## 1. 기본 정보
- 이름: [추가 입력 필요]
- 연락처: [추가 입력 필요]
- 이메일: [추가 입력 필요]
- 거주 지역: [추가 입력 필요]
- 포트폴리오 또는 GitHub: [추가 입력 필요]

## 2. 지원 분야
- 희망 직무: [추가 입력 필요]
- 희망 산업 또는 회사: [추가 입력 필요]

## 3. 핵심 요약
[지원 직무와 관련된 경력, 프로젝트, 교육 경험을 바탕으로 작성 필요]

예시:  
[추가 입력 필요] 분야에 관심을 가지고 [관련 경험 또는 프로젝트]를 수행했습니다.  
[핵심 역량 1]과 [핵심 역량 2]를 바탕으로 업무를 수행했으며, [구체적인 결과 또는 성과]를 달성했습니다.  
새로운 환경에서 빠르게 배우고 협업하며, 지원 직무에 기여하고자 합니다.

## 4. 핵심 역량
- **[역량 1]**: [관련 경험 또는 활용 수준 입력 필요]
- **[역량 2]**: [관련 경험 또는 활용 수준 입력 필요]
- **[역량 3]**: [관련 경험 또는 활용 수준 입력 필요]
- **[역량 4]**: [관련 경험 또는 활용 수준 입력 필요]
- **[역량 5]**: [관련 경험 또는 활용 수준 입력 필요]

## 5. 경력 및 프로젝트

### [경력 또는 프로젝트명]
- 기간: [추가 입력 필요]
- 역할: [추가 입력 필요]
- 주요 업무:
  - [수행한 업무 또는 해결한 문제]
  - [협업, 분석, 기획, 개발 등 구체적인 활동]
- 사용 기술 또는 도구: [추가 입력 필요]
- 결과 및 성과:
  - [성과 또는 결과 입력 필요]
  - [가능한 경우 기간, 비율, 인원, 금액 등 구체적인 수치 입력]

### [경력 또는 프로젝트명]
- 기간: [추가 입력 필요]
- 역할: [추가 입력 필요]
- 주요 업무:
  - [수행한 업무 입력 필요]
- 사용 기술 또는 도구: [추가 입력 필요]
- 결과 및 성과:
  - [성과 입력 필요]

## 6. 학력
- 학교명: [추가 입력 필요]
- 전공: [추가 입

### 지시가 구체적으로 작성되도록
- 모호한 지시와 구체적 지시의 결과물 비교

In [15]:
text = "우리 재생크림은 손상된 피부를 즉각적으로 재생시켜줍니다 가격도 괜찮아요"

result = ask("이글 어떻게 평가해?"+text)
print(result)

문장이 짧고 핵심은 전달되지만, 광고 문구로는 **효능 표현이 과장되거나 오해를 일으킬 가능성**이 있습니다.

### 아쉬운 점
- **“손상된 피부를 즉각적으로 재생시켜줍니다”**
  - ‘즉각적’, ‘재생’은 의약품처럼 치료·회복 효과를 보장하는 인상을 줄 수 있습니다.
  - 화장품이라면 근거가 충분하지 않을 경우 광고 심의나 표시·광고 규정에 문제가 될 수 있습니다.
- **“가격도 괜찮아요”**
  - 다소 구어적이고 주관적입니다. “합리적인 가격”처럼 다듬으면 더 신뢰감이 있습니다.
  - 가능하면 용량, 가격, 사용 기간 등 구체적인 정보를 함께 제시하는 편이 좋습니다.

### 다듬은 예시
**안전하고 무난한 표현**
> 건조하고 민감해진 피부를 촉촉하게 케어해주는 재생크림. 부담 없는 합리적인 가격으로 만나보세요.

**조금 더 판매 중심으로**
> 피부 장벽이 약해져 건조해진 피부에 깊은 보습을 더해주는 재생크림입니다. 효과적인 피부 케어를 합리적인 가격으로 경험해보세요.

**‘재생’ 표현을 줄인 버전**
> 외부 자극으로 민감해진 피부를 편안하게 진정시키고, 촉촉한 피부로 가꿔주는 크림입니다.

제품이 실제로 **피부 장벽 개선, 보습 지속, 진정 효과** 등의 시험 결과를 보유하고 있다면, 그 근거에 맞춰 “피부 장벽 강화에 도움”, “보습 개선에 도움”처럼 구체적으로 표현하는 것이 가장 좋습니다.


In [16]:
text = "우리 재생크림은 손상된 피부를 즉각적으로 재생시켜줍니다 가격도 괜찮아요"

result = ask(f"""
             다음 홍보 문구의 약점 3가지를 알려주고 각각 더 구체 적으로 고친 예시 3가지 정도 제시
             [홍보문구]
             {text}
             """)
print(result)

### 약점 1. 효능을 과장하고 의학적 오해를 줄 수 있음
“손상된 피부를 **즉각적으로 재생**시켜줍니다”는 치료·재생 효과를 확정적으로 표현해 과장 광고로 보일 수 있습니다. 화장품이라면 실제 근거가 있는 범위에서 “보습”, “진정”, “피부 장벽 케어”처럼 표현하는 편이 안전합니다.

**고친 예시**
1. “건조하고 민감해진 피부에 촉촉함을 더해 피부 장벽 케어를 돕습니다.”
2. “외부 자극으로 예민해진 피부를 편안하게 진정시키고 보습을 채워줍니다.”
3. “사용 후 당김을 줄이고, 거칠어진 피부를 부드럽고 촉촉하게 가꿔줍니다.”

---

### 약점 2. 제품의 특징과 사용 대상이 구체적이지 않음
“재생크림”이라는 이름만으로는 어떤 피부 고민에 적합한지, 어떤 성분과 제형을 사용하는지 알기 어렵습니다. 소비자가 제품의 필요성을 판단할 수 있도록 대상과 특징을 제시하는 것이 좋습니다.

**고친 예시**
1. “세라마이드와 판테놀을 함유해 건조하고 민감한 피부의 보습과 장벽 케어를 돕는 크림입니다.”
2. “끈적임은 줄이고 보습감은 채운 산뜻한 제형으로, 세안 후 당김이 심한 피부에 적합합니다.”
3. “붉어짐과 건조함으로 예민해진 피부를 위해 만든 데일리 보습 크림입니다.”

※ 실제 함유 성분과 시험 결과가 있는 경우에만 해당 내용을 사용해야 합니다.

---

### 약점 3. “가격도 괜찮아요”가 모호하고 설득력이 약함
가격이 얼마인지, 어떤 점에서 합리적인지 알 수 없습니다. 구체적인 가격이나 용량, 사용 기간, 할인 혜택 등을 제시하면 신뢰도가 높아집니다.

**고친 예시**
1. “50mL 기준 19,800원으로, 매일 부담 없이 사용할 수 있습니다.”
2. “한 번에 소량만 발라도 넉넉하게 사용할 수 있는 80mL 대용량 보습 크림입니다.”
3. “출시 기념으로 정상가 24,000원에서 19,900원에 만나보실 수 있습니다.”

---

### 종합 수정 예시

1. **“세라마이드와 판테놀을 함유해 건조하고 민감해진 피부의 보습과 

### 예시 보여주기 - few shot

In [18]:
review = "라포슈포제 화장품이 기대했던거보다 훨씬 좋았고 피부 당김도 들했고 피부에 광도 나는 거 같아요"
result = ask(f"다음 리뷰의 감정을 판단해줘{review}")
print(result)

긍정적인 감정입니다.  
기대 이상으로 제품에 만족했으며, 피부 당김이 줄고 피부에 광이 난다는 점을 긍정적으로 평가하고 있습니다.


In [ ]:
review = "라포슈포제 화장품이 기대했던거보다 훨씬 좋았고 피부 당김도 들했고 피부에 광도 나는 거 같아요"
result = ask(f"다음 리뷰의 감정을 판단해줘{review}")
print(result)

In [20]:
sentence = "너 어디가? 지금 어디가는데"

few_ex = f"""
다음 표준어 문장을 자연스러운 전라도 사투리로 바꿔주세요.

변환 규칙:
- 문장의 뜻은 바꾸지 마세요.
- 전라도에서 자주 사용하는 말투와 어미를 사용하세요.
- 너무 과장되거나 알아보기 어려운 표현은 피하세요.
- 설명하지 말고, 변환된 문장만 출력하세요.

# 예시
## 문장1
- 정말 맛있어요! -> 아따, 참말로 맛있당께!

## 문장2
- 지금 어디에 가요? -> 지금 어디 가는디?

## 문장3
- 빨리 와 주세요. -> 얼른 와 주랑께.

## 문장4
- 오늘 날씨가 정말 좋네요. -> 오늘 날씨가 참말로 좋구마잉.

## 문장5
- 그렇게 하면 안 돼요. -> 그렇게 하믄 안 된당께.

# 변환할 문장
{sentence}
"""

result = ask(few_ex)
print(result)

너 어디 가냐? 지금 어디 가는디?
